In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
import logging
import os

In [0]:
### Setup log

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger("bronze_football_pipeline")

##### Functions

In [0]:
### Standard Select

def get_standard_select() -> list:
    return [
        # match details
        F.col("match_id").cast("string"),
        F.col("match_date").cast("date"),
        F.col("kick_off").alias("kick_off"),
        F.col("home_score").cast("int"),
        F.col("away_score").cast("int"),

        # competition
        F.col("competition.competition_id").cast("int").alias("competition_id"),
        F.col("competition.competition_name").alias("competition_name"),
        F.col("competition.country_name").alias("country_name"),
        
        # season
        F.col("season.season_id").cast("int").alias("season_id"),
        F.col("season.season_name").alias("season_name"),
        
        # home_team
        F.col("home_team.home_team_id").alias("home_team_id"),
        F.col("home_team.home_team_name").alias("home_team_name"),

        # away_team
        F.col("away_team.away_team_id").alias("away_team_id"),
        F.col("away_team.away_team_name").alias("away_team_name"),

        F.col("match_status").alias("match_status"),
        F.col("match_status_360").alias("match_status_360"),
                            
        # Metadata
        F.col("_metadata.file_name").alias("_file_name"),
        F.col("_metadata.file_path").alias("_file_path"),
        F.current_date().alias("_load_dt"),
        F.current_timestamp().alias("_load_dttm")
    ]

In [0]:
### ingest_competition_bronze

def ingest_competition_bronze(
    competition_id: int,
    volume_base_dir: str = "/Volumes/workspace/football_project/raw_data",
    dest_schema: str = "football_project",
    force_refresh: bool = False
    ) -> None:
    
    # Check: comp_dir exists
    competition_dir = f"{volume_base_dir}/matches/{competition_id}"
    if not os.path.exists(competition_dir):
        logger.error(f"Competition directory does not exist: {competition_dir}")
        return

    # Check: already processed seasons
    matches_table = f"workspace.{dest_schema}.bronze_matches"
    if spark.catalog.tableExists(matches_table) and not force_refresh:
            existing_comp = (
                spark.table(matches_table)
                .filter(F.col("competition_id") == competition_id)
                .limit(1)
                .count()
            )
            if existing_comp > 0:
                logger.info(f"Competition {competition_id} already in Bronze. Skipping (Set force_refresh=True to re-ingest).")
                return

    # Check: season_dir exists
    season_list = [f"{competition_dir}/{s}" for s in os.listdir(competition_dir) if s.endswith(".json")]
    if not season_list:
        logger.error(f"NO SEASONS found for Competition ID: {competition_id}")
        return

    # Start the Process       
    logger.info(f"==================================================")
    logger.info(f"Starting ingestion for Competition ID: {competition_id} ({len(season_list)}) seasons")

    ### === bronze_matches ===
    matches_df = (
        spark.read
        .option("multiLine", True)
        .json(season_list)
        .select(*get_standard_select())
        )
    
    # Write to table
    (matches_df.write 
        .format("delta")
        .mode("overwrite")
        .partitionBy("competition_id")
        .option("replaceWhere", f"competition_id = {competition_id}")
        .option("mergeSchema", "true")
        .saveAsTable(f"workspace.{dest_schema}.bronze_matches")
    )

    logger.info(f"Saved bronze_matches lookup for Competition {competition_id}")

    ### === bronze_feeds ===

    # Prep 
    match_lookup_df = matches_df.select("match_id", "competition_id", "season_id").distinct()
    match_ids = [row.match_id for row in match_lookup_df.collect()]
    logger.info(f"Discovered {len(match_ids)} total matches.")

    # Loop: Feed > Ingest Feed
    for feed in ['events', 'lineups', 'three_sixty']:
        feed_paths = [
                f"{volume_base_dir}/{feed}/{m_id}.json"
                for m_id in match_ids 
                if os.path.exists(f"{volume_base_dir}/{feed}/{m_id}.json")
            ]
        
        # Check: feed_paths exists
        if not feed_paths:
            logger.warning(f" No files found for '{feed}'. Skipping.")
            continue
        
        logger.info(f"Found {len(feed_paths)} files for {feed.upper()}...")

        # Ingest: feed
        feed_df = (
            spark.read
            .option("multiLine", True)
            .json(feed_paths)
            .withColumn("match_id", F.regexp_extract(F.col("_metadata.file_name"), r"(\d+)", 1))
            .withColumn("_file_name", F.col("_metadata.file_name"))
            .withColumn("_file_path", F.col("_metadata.file_path"))
            .withColumn("_load_dttm", F.current_timestamp())
            .withColumn("_load_dt", F.current_date())
        )

        # Broadcast: "match_id", "competition_id", "season_id"
        broadcast_feed_df = feed_df.join(
                    F.broadcast(match_lookup_df),
                    on="match_id",
                    how="inner"      
                )
        
        table_name = f"workspace.{dest_schema}.bronze_{feed}"

        (
            broadcast_feed_df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("competition_id")
            .option("replaceWhere", f"competition_id = {competition_id}")
            .option("mergeSchema", "true")
            .saveAsTable(table_name)
        )
        logger.info(f"Successfully ingested into {table_name} [Competition: {competition_id}]")

    logger.info(f"Bronze Ingestion Complete for Competition {competition_id}!")


##### Run

In [0]:
ingest_competition_bronze(43)

2026-08-19 08:53:09 [INFO] Competition 43 already in Bronze. Skipping (Set force_refresh=True to re-ingest).


In [0]:
# spark.sql("DROP TABLE IF EXISTS workspace.football_project.bronze_matches")
# spark.sql("DROP TABLE IF EXISTS workspace.football_project.bronze_events")
# spark.sql("DROP TABLE IF EXISTS workspace.football_project.bronze_lineups")
# spark.sql("DROP TABLE IF EXISTS workspace.football_project.bronze_three_sixty")